# Using Kerchunk and Icechunk with grib2io and Xarray

This notebook demonstrates how to virtualize GRIB2 files using Kerchunk and store them in a cloud-native, versioned format using Icechunk.

In [ ]:
import grib2io
import xarray as xr
from grib2io.kerchunk import ReferenceGenerator
from grib2io.icechunk import IcechunkWriter
import os

## 1. Generate Kerchunk References
We use `ReferenceGenerator` to scan a GRIB2 file and create a manifest of byte-range references.

In [ ]:
# Use a local file for this demo
grib2_file = "sample.grib2"

# In a real scenario, this would be a large file or multiple files.
gen = ReferenceGenerator(grib2_file)
manifest = gen.generate()
print(f"Generated manifest with {len(manifest['refs'])} references")

## 2. Write to Icechunk
Icechunk provides a transactional, versioned storage layer for Zarr metadata.

In [ ]:
store_path = "my_icechunk_store"
writer = IcechunkWriter(store_path)
writer.write(manifest)
writer.commit("Initial ingest of GRIB2 data")

## 3. Open with Xarray
grib2io provides a custom backend for Xarray that can read directly from Icechunk stores.

In [ ]:
ds = xr.open_dataset(store_path, engine="grib2io", chunks={})
display(ds)

## 4. Efficient Variable Streaming
Because the data is lazily indexed, we can "stream" or access specific parts of a variable without loading the entire dataset into memory. Using `chunks={}` in `open_dataset` ensures that Xarray uses Dask for lazy loading.

This is particularly powerful for large variables like temperature (TMP) or humidity (RH) across multiple vertical levels or time steps.

In [ ]:
if "TMP" in ds.data_vars:
    # Select a subset of the variable (e.g., a specific region or level)
    # This only triggers byte-range requests for the relevant GRIB messages.
    tmp_subset = ds.TMP.isel(latitude=slice(0, 100), longitude=slice(0, 100))

    # Trigger a compute to see the data
    data = tmp_subset.compute()
    print("Successfully streamed TMP subset data.")
    display(data)

## 5. Multiple Variable Streaming
We can also stream multiple variables simultaneously. Dask will manage the parallel fetch of these chunks from the underlying storage (e.g. S3 or local disk) via the Icechunk references.

In [ ]:
# Select multiple variables for a specific region
variables_to_stream = ["TMP", "HGT"]
subset_ds = ds[variables_to_stream].isel(latitude=slice(50, 150), longitude=slice(50, 150))

# In a real-world scenario with many chunks, this would use dask's parallel execution.
computed_result = subset_ds.compute()
print("Streamed multiple variables (TMP, HGT) successfully.")